# Предсказание зарплаты по русским IT-вакансиям (hh.ru)

Мини-ресерч на базе семинара HSE NLP week 2 (TextCNN salary prediction).

**Идея:** тот же пайплайн, что в семинаре (токены → embedding → Conv1d → категории → `log1p(salary)`), но:
- датасет: IT-вакансии hh.ru в рублях;
- эмбеддинги: **Navec** вместо GloVe-twitter.

Полный train-скрипт: `train_salary_ru.py` (его удобнее гонять целиком). 



## 1. Данные и таргет

Таргет = зарплата в ₽. Если есть вилка `from`/`to`, берём среднее. `gross=True` грубо переводим «на руки» (`*0.87`). Выбросы режем.


In [ ]:
from pathlib import Path
import ast, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if not (ROOT / 'data' / 'hh_it_vacancies.csv').exists():
    ROOT = Path('week_02_salary_ru').resolve()

df_raw = pd.read_csv(ROOT / 'data' / 'hh_it_vacancies.csv')
print(df_raw.shape)
df_raw[['salary_from','salary_to','currency','gross']].head()


In [ ]:
def parse_skills(raw):
    if pd.isna(raw):
        return ''
    try:
        items = ast.literal_eval(str(raw))
        return ' '.join(x.get('name','') for x in items if isinstance(x, dict))
    except Exception:
        return re.sub(r'[^\w\s+/.-]', ' ', str(raw), flags=re.UNICODE)

def salary_rub(row):
    lo, hi = row['salary_from'], row['salary_to']
    if pd.isna(lo) and pd.isna(hi):
        return np.nan
    if pd.isna(lo):
        value = float(hi)
    elif pd.isna(hi):
        value = float(lo)
    else:
        value = 0.5 * (float(lo) + float(hi))
    if bool(row.get('gross')) is True:
        value *= 0.87
    return value

df = df_raw.copy()
df['skills_text'] = df['key_skills'].map(parse_skills)
df['Title'] = df['name'].fillna('').astype(str)
df['FullDescription'] = df['Title'] + ' ' + df['skills_text'] + ' ' + df['specializations_name'].fillna('').astype(str)
df['SalaryRub'] = df.apply(salary_rub, axis=1)
df = df[df['SalaryRub'].between(20_000, 400_000)].copy()
df['Log1pSalary'] = np.log1p(df['SalaryRub']).astype('float32')
for col in ['area_name','experience_name','schedule_name','employment_name']:
    df[col] = df[col].fillna('NaN').astype(str)
print('rows', len(df), 'median ₽', int(df.SalaryRub.median()))

plt.figure(figsize=(8,3))
plt.subplot(1,2,1); plt.hist(df.SalaryRub, bins=40); plt.title('SalaryRub')
plt.subplot(1,2,2); plt.hist(df.Log1pSalary, bins=40); plt.title('Log1pSalary')
plt.tight_layout()


## 2. Токенизация и словарь

Как в семинаре: `WordPunctTokenizer`, lowercase, `UNK`/`PAD`, слова с частотой ≥ 5.


In [ ]:
import nltk
from collections import Counter
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
tokenizer = nltk.tokenize.WordPunctTokenizer()

def tokenize_row(x):
    return ' '.join(tokenizer.tokenize(str(x))).lower()

for col in ['Title','FullDescription']:
    df[col] = df[col].map(tokenize_row)

token_counts = Counter()
for col in ['Title','FullDescription']:
    for text in df[col].values:
        token_counts.update(text.split())

tokens = sorted(t for t,c in token_counts.items() if c >= 5)
UNK, PAD = 'UNK', 'PAD'
tokens = [UNK, PAD] + tokens
token_to_id = {t:i for i,t in enumerate(tokens)}
print('vocab', len(tokens), 'top', token_counts.most_common(5))


## 3. Navec вместо GloVe

Строим embedding-матрицу `[vocab × 300]` из Navec. Слова вне Navec инициализируем случайно (как OOV в семинаре).


In [ ]:
from navec import Navec
import torch

navec_path = ROOT / 'artifacts' / 'navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(str(navec_path))
dim = int(navec.pq.dim)
emb_matrix = np.zeros((len(tokens), dim), dtype=np.float32)
hits = 0
for i, tok in enumerate(tokens):
    vec = navec.get(tok) or navec.get(tok.replace('ё','е'))
    if vec is not None:
        emb_matrix[i] = vec
        hits += 1
    else:
        emb_matrix[i] = np.random.normal(scale=0.6, size=(dim,)).astype(np.float32)
print(f'coverage {hits}/{len(tokens)} ({100*hits/len(tokens):.1f}%)')
emb_tensor = torch.tensor(emb_matrix)


## 4. Модель

Три ветки, как в семинаре:
1. Title → TextEncoder (Conv1d)
2. Description/skills → TextEncoder
3. categorical one-hot → Linear

Далее concat → Linear → `Log1pSalary`.

Для полного обучения на всех данных запускай:

```bash
python train_salary_ru.py
```

Он сохранит `artifacts/metrics.json`, `salary_ru_best.pt`, `demo_predictions.json`.


In [ ]:
from IPython.display import display
import json

metrics_path = ROOT / 'artifacts' / 'metrics.json'
demo_path = ROOT / 'artifacts' / 'demo_predictions.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('best MAE ₽:', round(metrics['best_mae_rub']))
    print('last epoch:', metrics['history'][-1])
else:
    print('metrics.json пока нет — сначала запусти train_salary_ru.py')

if demo_path.exists():
    demo = json.loads(demo_path.read_text())
    display(pd.DataFrame(demo))
